# CSV Basics — 07: CSV vs TSV and Other Delimiters

## What you will learn
CSV is just one of many delimiter-separated value formats.
Understanding the trade-offs helps you choose the right format
and handle edge cases.

In this notebook:
1. CSV vs TSV — when to use each
2. Semicolon-separated (European CSV)
3. The quoting problem — what happens when data contains the delimiter
4. RFC 4180 compliance — the "standard" that nobody fully follows
5. Custom delimiters and exotic formats
6. Auto-detecting the delimiter


In [1]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 09:10:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/csv_basics


In [2]:
import pathlib

# Sample data with content that contains commas and tabs
data = [
    (1, "Smith, John",    "New York, NY",       "Software Engineer, Senior", 95000.0),
    (2, "O'Brien, Mary",  "San Francisco, CA",  "Data Scientist",            105000.0),
    (3, "García, José",   "Miami, FL",          "Product Manager, Lead",     115000.0),
    (4, "Wang	Mei",      "Seattle, WA",        "DevOps Engineer",           98000.0),  # tab in name
]

schema_str = "id INT, name STRING, location STRING, title STRING, salary DOUBLE"

df = spark.createDataFrame(data, schema_str)
df.show(truncate=False)
print("Notice: names and locations contain COMMAS — this causes problems in CSV!")

[Stage 1:===================>                                       (1 + 2) / 3]

+---+-------------+-----------------+-------------------------+--------+
|id |name         |location         |title                    |salary  |
+---+-------------+-----------------+-------------------------+--------+
|1  |Smith, John  |New York, NY     |Software Engineer, Senior|95000.0 |
|2  |O'Brien, Mary|San Francisco, CA|Data Scientist           |105000.0|
|3  |García, José |Miami, FL        |Product Manager, Lead    |115000.0|
|4  |Wang\tMei    |Seattle, WA      |DevOps Engineer          |98000.0 |
+---+-------------+-----------------+-------------------------+--------+

Notice: names and locations contain COMMAS — this causes problems in CSV!


## Step 1 — The CSV Quoting Problem

When values contain the delimiter (comma), they MUST be quoted.
When values contain quotes, they must be escaped (doubled or backslash).


In [3]:
# CSV — commas inside values REQUIRE quoting
csv_path = f"{DATA_DIR}/employees.csv"
df.coalesce(1).write.mode("overwrite").csv(csv_path, header=True, quote='"')

import glob
csv_file = glob.glob(f"{csv_path}/part-*.csv")[0]
print("CSV output (commas in values are quoted):")
with open(csv_file) as f:
    print(f.read())

# TSV — tabs are the delimiter, commas in values are safe (no quoting needed)
tsv_path = f"{DATA_DIR}/employees.tsv"
df.coalesce(1).write.mode("overwrite").csv(tsv_path, header=True, sep="\t", quote='"')

tsv_file = glob.glob(f"{tsv_path}/part-*.csv")[0]
print("\nTSV output (commas fine, but tab in Wang\tMei causes a problem!):")
with open(tsv_file) as f:
    print(f.read())

CSV output (commas in values are quoted):
id,name,location,title,salary
1,"Smith, John","New York, NY","Software Engineer, Senior",95000.0
2,"O'Brien, Mary","San Francisco, CA",Data Scientist,105000.0
3,"García, José","Miami, FL","Product Manager, Lead",115000.0
4,Wang	Mei,"Seattle, WA",DevOps Engineer,98000.0




TSV output (commas fine, but tab in Wang	Mei causes a problem!):
id	name	location	title	salary
1	Smith, John	New York, NY	Software Engineer, Senior	95000.0
2	O'Brien, Mary	San Francisco, CA	Data Scientist	105000.0
3	García, José	Miami, FL	Product Manager, Lead	115000.0
4	"Wang	Mei"	Seattle, WA	DevOps Engineer	98000.0



In [4]:
# Read them back and verify
df_csv = spark.read.csv(csv_path, header=True, sep=",",  quote='"', inferSchema=True)
df_tsv = spark.read.csv(tsv_path, header=True, sep="\t", quote='"', inferSchema=True)

print("CSV read back:")
df_csv.show(truncate=False)

print("\nTSV read back:")
df_tsv.show(truncate=False)
print("\nNote: row 4 (Wang\tMei) has tab in name — corrupted in TSV!")

CSV read back:
+---+-------------+-----------------+-------------------------+--------+
|id |name         |location         |title                    |salary  |
+---+-------------+-----------------+-------------------------+--------+
|1  |Smith, John  |New York, NY     |Software Engineer, Senior|95000.0 |
|2  |O'Brien, Mary|San Francisco, CA|Data Scientist           |105000.0|
|3  |García, José |Miami, FL        |Product Manager, Lead    |115000.0|
|4  |Wang\tMei    |Seattle, WA      |DevOps Engineer          |98000.0 |
+---+-------------+-----------------+-------------------------+--------+


TSV read back:
+---+-------------+-----------------+-------------------------+--------+
|id |name         |location         |title                    |salary  |
+---+-------------+-----------------+-------------------------+--------+
|1  |Smith, John  |New York, NY     |Software Engineer, Senior|95000.0 |
|2  |O'Brien, Mary|San Francisco, CA|Data Scientist           |105000.0|
|3  |García, José |

## Step 2 — Semicolon-Separated: European CSV

In European locales, Excel uses semicolon as separator because
comma is used as decimal separator (1.000,50 instead of 1,000.50).


In [5]:
# European CSV with semicolons and European number format
euro_csv = """id;name;amount;date
1;Müller, Hans;1.234,56;15/01/2024
2;Dubois, Marie;2.567,89;16/01/2024
3;García, Pedro;3.890,12;17/01/2024"""

euro_path = f"{DATA_DIR}/european.csv"
pathlib.Path(euro_path).write_text(euro_csv, encoding="UTF-8")

# Read with semicolon separator
df_euro_raw = spark.read.csv(euro_path, header=True, sep=";", encoding="UTF-8")
print("European CSV (raw — amounts are strings with European format):")
df_euro_raw.show(truncate=False)
print("Schema:", df_euro_raw.dtypes)
print()

# Fix: convert European number format to standard
from pyspark.sql.functions import regexp_replace, col

df_euro_fixed = df_euro_raw.withColumn(
    "amount",
    regexp_replace(regexp_replace(col("amount"), "\\.", ""), ",", ".").cast("double")
).withColumn(
    "date",
    F.to_date(col("date"), "dd/MM/yyyy")
)
print("After normalizing European number format:")
df_euro_fixed.show(truncate=False)

European CSV (raw — amounts are strings with European format):
+---+-------------+--------+----------+
|id |name         |amount  |date      |
+---+-------------+--------+----------+
|1  |Müller, Hans |1.234,56|15/01/2024|
|2  |Dubois, Marie|2.567,89|16/01/2024|
|3  |García, Pedro|3.890,12|17/01/2024|
+---+-------------+--------+----------+

Schema: [('id', 'string'), ('name', 'string'), ('amount', 'string'), ('date', 'string')]

After normalizing European number format:
+---+-------------+-------+----------+
|id |name         |amount |date      |
+---+-------------+-------+----------+
|1  |Müller, Hans |1234.56|2024-01-15|
|2  |Dubois, Marie|2567.89|2024-01-16|
|3  |García, Pedro|3890.12|2024-01-17|
+---+-------------+-------+----------+



## Step 3 — RFC 4180: The CSV "Standard"

RFC 4180 defines the rules for CSV. Most tools claim to follow it but don't.
Knowing the rules helps you handle edge cases.


In [6]:
# RFC 4180 key rules:
# 1. Each record on separate line ending with CRLF (\r\n)
# 2. Last record MAY or MAY NOT have ending line break
# 3. First line MAY be header
# 4. Fields MAY be enclosed in double quotes
# 5. Fields with special chars MUST be enclosed in double quotes
# 6. Double-quotes in fields must be escaped by doubling ("")

rfc_csv = '''id,value,description
1,100,"Simple value"
2,200,"Value with, comma"
3,300,"Value with ""quoted"" text"
4,400,"Value with
newline"
'''

rfc_path = f"{DATA_DIR}/rfc4180.csv"
pathlib.Path(rfc_path).write_bytes(rfc_csv.encode('utf-8'))

print("RFC 4180 compliant CSV with edge cases:")
df_rfc = spark.read.csv(
    rfc_path,
    header=True,
    quote='"\'',
    escape='"\'',      # doubled quote = escape (RFC 4180 rule 6)
    multiLine=True,   # needed for newlines inside quoted fields
    inferSchema=True
)
df_rfc.show(truncate=False)
print("All edge cases handled correctly with proper options")

RFC 4180 compliant CSV with edge cases:


SparkRuntimeException: quote cannot be more than one character.

## Step 4 — Auto-Detecting the Delimiter


In [ ]:
def detect_delimiter(filepath, sample_lines=5):
    """
    Auto-detect CSV delimiter by counting occurrence of common delimiters
    in the first few lines.
    """
    with open(filepath) as f:
        sample = [f.readline() for _ in range(sample_lines)]

    candidates = {",": 0, ";": 0, "\t": 0, "|": 0}

    for line in sample:
        for delim in candidates:
            # Count occurrences but don't count inside quotes
            in_quotes = False
            for char in line:
                if char == '"':
                    in_quotes = not in_quotes
                elif char == delim and not in_quotes:
                    candidates[delim] += 1

    best = max(candidates, key=candidates.get)
    counts = {k: v for k, v in candidates.items() if v > 0}
    return best, counts

# Test on different files
test_files = [
    (csv_path,   "employees.csv (comma)"),
    (tsv_path,   "employees.tsv (tab)"),
    (euro_path,  "european.csv (semicolon)"),
]

print("Auto-detecting delimiters:")
for path, label in test_files:
    files = glob.glob(f"{path}/part-*.csv")
    if files:
        detected, counts = detect_delimiter(files[0])
    else:
        detected, counts = detect_delimiter(path)
    delim_name = {",":"comma", ";":"semicolon", "\t":"tab", "|":"pipe"}.get(detected, detected)
    print(f"  {label:<35} → {delim_name} | counts: {counts}")

## Summary: Delimiter Format Guide

| Format | Delimiter | Use when | Watch out for |
|---|---|---|---|
| CSV | `,` | Universal exchange | Values with commas need quoting |
| TSV | `\t` | When values have commas | Values with tabs are corrupted |
| Semicolon | `;` | European locale Excel | Values with semicolons need quoting |
| Pipe | `\|` | When values have commas and semicolons | Values with pipes need quoting |

**Rule:** Choose a delimiter that does NOT appear in your data values.
If values can contain any printable character, use a non-printable delimiter
like `\x01` (SOH) or `\x1f` (Unit Separator).

```python
# Non-printable delimiter
df.write.csv(path, sep="\x01")
spark.read.csv(path, sep="\x01")
```
